# 🌊 Underwater Image Enhancement + Marine Life Detection Demo

This notebook demonstrates the complete pipeline for:
1. **Stage 1**: Image enhancement (CLAHE, white balance, dehazing)
2. **Stage 2**: Marine life detection using YOLOv5

## Setup

First, let's install dependencies and import required modules.

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add project root to path
sys.path.insert(0, '.')

# Import our modules
from denoising.clahe import apply_clahe, adaptive_clahe
from denoising.white_balance import gray_world, combined_white_balance
from denoising.dehazing import underwater_dehaze, dark_channel_prior

print("✓ Modules imported successfully")

## Helper Functions

In [ ]:
def show_images(images, titles, figsize=(15, 5)):
    """Display multiple images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for ax, img, title in zip(axes, images, titles):
        # Convert BGR to RGB for display
        if len(img.shape) == 3:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        else:
            img_rgb = img
        ax.imshow(img_rgb)
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

def create_sample_underwater_image():
    """Create a simulated underwater image for demonstration."""
    # Create a gradient blue-green image simulating underwater
    h, w = 480, 640
    image = np.zeros((h, w, 3), dtype=np.uint8)
    
    # Blue-green gradient (underwater color cast)
    for y in range(h):
        blue = int(150 + (y / h) * 50)
        green = int(80 + (y / h) * 40)
        red = int(30 + (y / h) * 20)  # Red attenuated
        image[y, :] = [blue, green, red]
    
    # Add some "objects" (circles simulating fish)
    np.random.seed(42)
    for _ in range(5):
        cx = np.random.randint(50, w-50)
        cy = np.random.randint(50, h-50)
        radius = np.random.randint(20, 50)
        color = (
            np.random.randint(100, 200),
            np.random.randint(50, 150),
            np.random.randint(30, 100)
        )
        cv2.circle(image, (cx, cy), radius, color, -1)
    
    # Add haze effect
    haze = np.ones_like(image) * np.array([180, 120, 60], dtype=np.uint8)
    image = cv2.addWeighted(image, 0.6, haze, 0.4, 0)
    
    # Add noise
    noise = np.random.normal(0, 15, image.shape).astype(np.int16)
    image = np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return image

print("✓ Helper functions defined")

## 1. Load or Create Sample Image

Either load your own underwater image or use a generated sample.

In [ ]:
# Option 1: Create sample image (for demo)
image = create_sample_underwater_image()

# Option 2: Load your own image
# image = cv2.imread('your_underwater_image.jpg')

show_images([image], ['Original Underwater Image'])
print(f"Image shape: {image.shape}")

## 2. CLAHE Enhancement

Contrast Limited Adaptive Histogram Equalization improves local contrast.

In [ ]:
# Apply CLAHE with different settings
clahe_default = apply_clahe(image, clip_limit=2.0)
clahe_strong = apply_clahe(image, clip_limit=4.0)
clahe_adaptive = adaptive_clahe(image)

show_images(
    [image, clahe_default, clahe_strong, clahe_adaptive],
    ['Original', 'CLAHE (2.0)', 'CLAHE (4.0)', 'Adaptive CLAHE'],
    figsize=(20, 5)
)

## 3. White Balance Correction

Correct the blue-green color cast typical of underwater images.

In [ ]:
from denoising.white_balance import shades_of_gray, white_patch_retinex

# Apply different white balance methods
wb_gray_world = gray_world(image)
wb_shades = shades_of_gray(image, p=6.0)
wb_white_patch = white_patch_retinex(image)
wb_combined = combined_white_balance(image, method='auto')

show_images(
    [image, wb_gray_world, wb_shades, wb_combined],
    ['Original', 'Gray World', 'Shades of Gray', 'Auto (Combined)'],
    figsize=(20, 5)
)

## 4. Underwater Dehazing

Remove haze/turbidity from underwater images.

In [ ]:
from denoising.dehazing import simple_dehaze

# Apply dehazing methods
dehazed_dcp = dark_channel_prior(image)
dehazed_underwater = underwater_dehaze(image)
dehazed_simple = simple_dehaze(image, strength=0.7)

show_images(
    [image, dehazed_dcp, dehazed_underwater, dehazed_simple],
    ['Original', 'Dark Channel Prior', 'Underwater Dehaze', 'Simple Dehaze'],
    figsize=(20, 5)
)

## 5. Combined Enhancement Pipeline

Apply all enhancement techniques in sequence for best results.

In [ ]:
def full_enhancement(image):
    """Apply complete enhancement pipeline."""
    # Step 1: White balance correction
    step1 = combined_white_balance(image, method='auto')
    
    # Step 2: Adaptive CLAHE
    step2 = adaptive_clahe(step1)
    
    # Step 3: Underwater dehazing
    step3 = underwater_dehaze(step2)
    
    return step1, step2, step3

step1, step2, enhanced = full_enhancement(image)

show_images(
    [image, step1, step2, enhanced],
    ['Original', 'Step 1: White Balance', 'Step 2: CLAHE', 'Step 3: Dehazed'],
    figsize=(20, 5)
)

# Final comparison
show_images(
    [image, enhanced],
    ['Original', 'Fully Enhanced'],
    figsize=(12, 5)
)

## 6. Marine Life Detection

Use YOLOv5 to detect marine life in enhanced images.

In [ ]:
try:
    from detection.detect import MarineDetector
    
    # Initialize detector (uses pretrained COCO model if no custom model)
    detector = MarineDetector(
        confidence_threshold=0.25,
        iou_threshold=0.45
    )
    
    print("✓ Detector initialized")
    
    # Run detection on enhanced image
    detections, annotated = detector.detect(enhanced, visualize=True)
    
    # Display results
    show_images([enhanced, annotated], ['Enhanced', 'Detections'])
    
    print(f"\nFound {len(detections)} objects:")
    for det in detections:
        print(f"  - {det['class_name']}: {det['confidence']:.2f}")
        
except ImportError as e:
    print(f"Detection module not available: {e}")
    print("Install ultralytics: pip install ultralytics")

## 7. Full Pipeline Demo

Use the integrated pipeline for end-to-end processing.

In [ ]:
try:
    from pipeline.run_pipeline import UnderwaterPipeline, PipelineConfig
    
    # Configure pipeline
    config = PipelineConfig(
        enhancement_method='combined',
        detection_enabled=True,
        detection_confidence=0.25,
        save_enhanced=True,
        save_annotated=True
    )
    
    # Initialize pipeline
    pipeline = UnderwaterPipeline(config)
    
    # Process image
    result = pipeline.process_image(image)  # Pass array directly
    
    print(f"Processing time: {result.processing_time:.2f}s")
    print(f"Detections: {len(result.detections)}")
    
    # Display results
    show_images(
        [image, result.enhanced_image, result.annotated_image],
        ['Original', 'Enhanced', 'Detected'],
        figsize=(18, 5)
    )
    
except Exception as e:
    print(f"Pipeline error: {e}")

## 8. Save Results

In [ ]:
# Create output directory
output_dir = Path('demo_output')
output_dir.mkdir(exist_ok=True)

# Save images
cv2.imwrite(str(output_dir / 'original.jpg'), image)
cv2.imwrite(str(output_dir / 'enhanced.jpg'), enhanced)

print(f"✓ Results saved to {output_dir}/")
print(f"  - original.jpg")
print(f"  - enhanced.jpg")

## 9. Metrics Comparison

In [ ]:
def calculate_image_stats(image):
    """Calculate image quality metrics."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    return {
        'mean_brightness': np.mean(gray),
        'std_contrast': np.std(gray),
        'mean_r': np.mean(image[:, :, 2]),
        'mean_g': np.mean(image[:, :, 1]),
        'mean_b': np.mean(image[:, :, 0]),
    }

# Compare original vs enhanced
orig_stats = calculate_image_stats(image)
enh_stats = calculate_image_stats(enhanced)

print("Image Quality Metrics:")
print("=" * 50)
print(f"{'Metric':<20} {'Original':>12} {'Enhanced':>12}")
print("-" * 50)
for key in orig_stats:
    print(f"{key:<20} {orig_stats[key]:>12.2f} {enh_stats[key]:>12.2f}")

## Summary

This demo showed:

1. **CLAHE Enhancement**: Improves local contrast
2. **White Balance**: Corrects underwater color cast
3. **Dehazing**: Removes turbidity and haze
4. **Combined Pipeline**: All enhancements in sequence
5. **Detection**: YOLOv5-based marine life detection

For real underwater images, try adjusting parameters for best results!